# ML 工程：一次准备，保存信号，多次账户消费

**叙事与执行预算初稿，尚未执行本 Notebook。** 这一页用于先审设计，收益与参数优劣不作为工程通过条件。全年和五年构建暂停，必须先共同验收设计与教学小流程。

正文合同仍归 [Research](../docs/design/05_axiom_research.md)、[Core](../docs/design/03_axiom_core.md)、[Engine](../docs/design/04_axiom_trade.md)。本 Notebook 是连贯的可执行教学入口，不建立第二份接口规范。

## 1. 整体图：谁拥有输入、计算与输出

![已实现架构及规划边界](../docs/design/ml-engineering-assets/architecture.svg)

实线表示已有固定样本代码及消费链；虚线表示仍需完成的通用合同。用户已确认功能方向与基础 UI，不能因此把 ModelHandle、通用 SignalPlan、多策略组合或正式 rolling 恢复画成已经完成。inference 当前在 Research；账户与评估只由 Engine 执行。

### 两项需要先裁决的架构风险

| 现状与风险 | 建议目标 | 小例如何验 |
|---|---|---|
| Research builder 把 Research Python、Data Python 与 Core Python 的整体实现 digest 纳入身份；非语义代码改动也可能 MISS，保存输入再利用仍限同配置 | 后续按实际计算依赖与语义版本拆分身份；保留完整实现来源用于审计。变更前先冻结哪些修改应失效，避免默默宽松复用 | 同输入加载旧保存件仍通过；当前 MISS 如实报告。先列失效矩阵，教程不直接改生产 cache |
| 长历史 Feature / proof 大对象可能一直驻留；两折小例未证明多年内存有界 | 按完整日期截面分片持久化、逐片校验并释放；固定 parent/ref 图与原子 checkpoint | 先完整一周、再代表范围测 RSS / 文件字节 / 中断恢复；未通过不扩全年 |

状态：这两项是待父任务裁决的目标建议，不作为已经合理且完成的设计。
四项候选的准确边界统一见 [Research §8.4.1](../docs/design/05_axiom_research.md#ml-engineering-review-proposals)：阶段依赖身份保留实际计算实现与完整审计；首轮按两个完整日期分片、原 proof blob 独立保存并每次核验。它们尚未实施，本教学不先大改缓存或 loader。


### SysQ：借经验，明确已有差异

| 已读证据中的 SysQ 做法 | Axiom 当前实现与取舍 |
|---|---|
| Feature per-window write-through；命中后读原窗口 frame，再训练 | 已有固定单折保存输入复用。本轮跨 fold 的完整日期 parent 编排是有界验收脚本，尚非公开 rolling 缓存平台 |
| SignalStore / LabelStore / 实验索引；信号分析与 signal-driven backtest 分开 | 保存 Signal 后交 Engine 唯一账户；只读 UI 使用 owner 保存件 |
| 已读 research usecase 的成熟门槛与 lineage 检查仍列 TBD | 当前逐行成熟时钟、输入 digest、完整 universe / invalid、分数阶段已明确保留；不沿用工作日 fallback、填零或当前成员替代历史 |

此表沿用 [此前固定 SysQ 对照](../docs/design/05_axiom_research.md#stock-qlib-lightgbm-minimal) 中的 usecase / generator / training 证据，不是新一轮全仓审计或 SysQ 当前全量评价。

## 2. 固定真实短样本：声明输入，再开始演示

教学主线复用原 January 保存件与原冻结账户输入。它是 314 ID 的固定预测 union、六特征和真实实际交易日；执行范围仍是已准入的 SZSE 主板子集。完整来源、成员、cutoff 与缺失限制随 owner 保存件读取，不用当前成员替代历史。

公开代码仅接受调用者环境变量中的保存件目录和新的输出目录；不提交本机路径、token、原始数据或完整私有输入。每阶段输出小表与一句解释，完整 refs/证明留在本地验收 sidecar。

In [ ]:
import os
from pathlib import Path
from axiom_research import load_stock_ml_experiment

experiment_path = Path(os.environ["AXIOM_ML_EXPERIMENT_DIR"])
output_path = Path(os.environ["AXIOM_ML_TUTORIAL_OUTPUT"])
experiment = load_stock_ml_experiment(experiment_path)
# 展示范围、规模与来源限制；这一步是保存件 load / verify，不是 Data 冷准备。


## 3. Feature ID 与依赖：表从 registry 自动生成

下面表来自已合 Research 的 `stock_ml_v1.json`，不另维护手写公式源。ID 按语义分组，不要求连续；列顺序由本模型实际 feature_selection 决定。正式执行时再从公共 catalog 与保存模型选择自动核对。

| 分组 | ID / 名称 | 公式或含义 | 依赖字段 | Lookback / session | 语义版本 | 本模型 |
|---|---|---|---|---|---|---|
| MOM | MOM010 / ret1 | adjusted_close[t] / adjusted_close[t-1] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 2 | 1.0.0 | 选用 |
| MOM | MOM020 / ret5 | adjusted_close[t] / adjusted_close[t-5] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 6 | 1.0.0 | 选用 |
| MOM | MOM030 / ret20 | adjusted_close[t] / adjusted_close[t-20] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 21 | 1.0.0 | 选用 |
| VOL | VOL010 / intraday_range | (high[t] - low[t]) / close[t] | market_daily.high, market_daily.low, market_daily.close, adjustment_factors.factor | 1 | 1.0.0 | 选用 |
| PRC | PRC010 / close_to_open | close[t] / open[t] - 1 | market_daily.close, market_daily.open, adjustment_factors.factor | 1 | 1.0.0 | 选用 |
| LIQ | LIQ010 / amount_relative_5 | amount_cny[t] / rolling_mean(amount_cny, window=5, inclusive_current=true) | market_daily.amount_cny | 5 | 1.0.0 | 选用 |

[固定 catalog 源](https://github.com/sinnergarden/axiom-research/blob/90b378a6e7d8283b0b021c537a95c5468b9161bc/src/axiom_research/catalogs/stock_ml_v1.json)。

## 4. Data / Qlib / FeatureBuild：首次与复用是不同成本

首次准备由 Data 将 Raw 规范为不可变 Snapshot。公开 QuerySpec 冻结字段、成员、日历、PIT 与逐 session cutoff；Qlib 导出只生成格式数值与完整 manifest / 证明。Research 把 catalog 编译成 Core FeaturePlan，使用共享执行器生成 FeatureBuild。

本教学主线先加载已有 FeatureBuild 与原 Data/Qlib 来源证明，不重采集、不冒称新冷准备。若父任务批准轻量 Core 单日期示例，将单列实际计算与其完整截面范围。Qlib 复用仍要完整文件 / 来源核验；后端证明解析不是零 I/O 或零计算。

## 5. 五 session label：成熟之后才能进入训练

标签以真实交易日历的 f+1 开盘、f+5 收盘计算；fit clock 之外的 outcome 不得进入训练。归一化只在同一天当时成员、有限值且已成熟的完整截面上 cs_zscore（ddof=0、无 clip）；不是全期间拟合变换。

正式运行将显示一个真实日期的小例：feature 日、实际入场/结束日、label available_at、fit cutoff、截面人数、均值/标准差与保存标准化值。January 原拟合时钟为 2023-12-28 20:30 +08，最后成熟 Feature 日为 12-21；不会写成 12 月完整月末重训。

## 6. Dataset 与模型：fit 毫秒，准备和证明并非毫秒

公开 `build_stock_ml_from_saved_features` 可在固定同配置输入上重建归一化 label / Dataset / 模型；不读 Data、不重算 Feature。第一次新目标显式记录 normalization / fit / infer / serialization；第二次完全相同目标实际调用 builder，必须 HIT 且 fit / infer / Data / FeatureCore 均为零。

每次变更 fit 窗口 / cutoff 的 weekly 实验不满足此 helper 的同配置条件。weekly 小样本只能在批准的有界编排中做，不能给它套上通用公开平台名称。

In [ ]:
from axiom_research import build_stock_ml_from_saved_features

first_metrics, second_metrics = {}, {}
first = build_stock_ml_from_saved_features(
    experiment_path, destination=output_path / "same-input-model", metrics=first_metrics)
second = build_stock_ml_from_saved_features(
    experiment_path, destination=output_path / "same-input-model", metrics=second_metrics)
# 正式执行将显示两次计时、cache_hit 与实际调用数；不打印整份 JSON。


## 7. 保存 Signal：预测分数与账户目标分开

模型生成 prediction_raw 的无量纲分数；保存 Signal 保留 model / Feature 引用、完整键、成员、validity 和模拟可用时点。Signal 不包含账户持仓路径或撮合结果。Engine 使用严格前一实际 session 的保存预测，不回填未来模型分数。

正式执行核对保存与重新加载的 Signal 身份，再交 Engine 中立 StockPredictionFrame。纯预测信号可以跨独立账户复用；依赖持仓路径的策略状态不能跨账本复用缓存。

## 8. Engine 唯一账户 / Evaluation：同 Signal 换组合

Core 将保存预测与当前账户 / 时钟结合，Runtime 唯一生成意图、成交、现金、持仓与 NAV，Evaluation 消费保存 run 与固定基准 / 行动范围。Research 和 UI 都不另写撮合或净值公式。

**当前真实缺口：** Engine 公共股票合同严格固定 `top_k=5`，Core context 无 TopK 参数；目前无法完成不同 TopK 的公共对照。父任务需裁决最小 Engine 合同增量（例：Top5 与 Top3、其准入/排名/预算语义），本 Docs 任务不修改 Engine，也不以改初始资金冒充改策略。

增量通过后，将从同一保存 Signal 各运行两个不同组合配置。执行计数探针必须证明 Data / supplier / Feature / fit / predict = 0，Signal 身份不变；仅 Core 组合、Runtime、Evaluation 实际执行。原 run / evaluation 只读，新结果独立保存。

待审最小接口建议是保持 `BacktestRequest.from_dict` / `run_backtest`，先只允许 `portfolio_policy.top_k ∈ {3,5}`，Core 增加 `plan_stock_portfolio(frame, *, account, context, top_k=5)`。K 进入决策/意图身份及 trace；不足 K 个有效成员为 NO_DECISION，等权预算/K，原执行池、时钟、费用、T+1 和 Unknown 政策均不改。旧明确 K5 请求与保存文件保留。准确合同由 Engine owner 汇合，本 PR 不编辑 Trade 主章。

真正对照固定同一 January Signal/market/初始账户/profile，实际 K5 与 K3 至少有一个不同目标组合；同时探针证明 Data/supplier/Feature/fit/predict 零调用。它仅适用于无账户依赖的预测 Signal；不同账户路径与策略状态不共享缓存。


## 9. UI：保存值进入同一个网页入口

UI 公共 Reader 校验 owner 保存件后生成展示投影；已导出 HTML 打开和切页不执行 Data、模型或账户。GitHub Pages 是今晚统一入口；此 Notebook 生成 HTML 后由既有 Pages 发布流程链接，Docs 任务不修改 UI 源或自行发布。

功能已确认、源码已实现、有界运行通过、UI owner 浏览器验证、用户最终验收分别记录。当前这份教学仅为设计/预算初稿，不冒称教学已跑通。

## 10. 重跑边界与已有测量：每种成本分开看

| 改动 / 动作 | 应复用 | 实际需要执行 / 当前限制 |
|---|---|---|
| 只打开已导出 HTML | 所有 owner 结果 | 仅浏览器；不跑 Reader / train / account |
| 加载保存模型 / Signal / run | 原结果内容 | 解析、hash/ref/证明核验；不是零成本 |
| 完全相同 builder 目标 | 全实验 | 公共调用命中后 Data / Core / fit / predict 为 0；仍完整加载核验 |
| 同一独立预测 Signal 改组合 | Data / Feature / Model / Signal | Core 组合 / Runtime / Evaluation；当前 TopK 增量尚缺，不能声称演示已通过 |
| 改 label / 模型语义 | 未受影响 Feature / 原事实 | 新 Dataset / model / Signal；现公开保存输入 helper 限同 config |
| 改 Feature 公式 / 字段、Snapshot、PIT/cutoff | 仅实际未受影响阶段 | 新语义链；新增字段或导出 query 变化需新 Qlib view |
| 非语义 Python 修改 | 旧保存件仍可读 | wholeimplementation 可能过粗 MISS；不能报告为语义缓存能力 |

| 既有真实测量，未在本稿重跑 | 观察值 | 可用于解释的范围 |
|---|---|---|
| Data Qlib 314 ID × 56 sessions 冷准备（旧 exporter） | 37.72 s；峰值 5.43 GiB | 独立格式准备；不含模型 / Feature / 账户，新 Data PR39 无生产耗时 / RSS 实测 |
| 有界 monthly 两折 | 282.34 s；峰值 4.09 GiB | February 完整 15 日，March 仅 3/21 日；含额外等价核验，不是 weekly 或全年 |
| 其中 LightGBM fit / infer | fit 61.94 / 42.44 ms；infer 7.79 / 1.58 ms | 单阶段；矩阵/label/来源证明/序列化另算 |
| Engine 一组完整保存账户 + 评价加载 | 27.34 → 14.90 s，下降 45.5% | Reader GROUP 单样本；不是整个网页导出/浏览器速度 |
| 新 Data 导出 manifest 次数 | 冷 5→1；warm 每次重新核验 1 | 算法调用数，不能称 5 倍提速 |

OS 页缓存未清空，单阶段时间有嵌套，不能相加为另一个总时间；没有全年吞吐保证。

## 11. Weekly 代表小样本与执行预算：审后再跑

用户要求的是未来五年 **weekly retrain**，约 260 folds 的数量只是参考，必须按真实日历决定；旧 monthly 3–6 小时外推失效，不用作 weekly 结论。当前不启动全年或五年。

先拟两个真实 weekly windows：2024-02-05—02-08、02-19—02-23，分别使用严格前一 session 02-02 / 02-08 的 fit clock。教学控制训练 Feature 窗口明确为 **2023-11-01—2024-01-31**，按该周 20:30 +08 的 fit 时钟重新筛成熟键；314 原 January ID 是规模对照，不能称这两周完整 CSI300。每周预测保留严格前 session anchor，按完整日期截面生成，禁止按证券切碎 normalization。

| 阶段 | 拟实际执行 | 初始上限 / 必须保存证据 |
|---|---|---|
| 首次准备证据 | 加载固定 Data/Qlib/Feature 来源；不重新冷导出 | 计解析/核验耗时、RSS、bytes；冷准备只引用已有测量 |
| 单折教学 + 完全重复 | 同输入 saved-Feature builder 首次与第二次；最多 1 次业务 fit / predict | 每阶段计时 / 调用计数；cache HIT 必须真实调用 |
| 两个 weekly fold | 仅缺失 label outcome 的公共 cutoff 查询、成熟 Dataset/norm、2 次 fit / infer、保存 I/O | 拟至多 8 次公共 Data 查询；缺口或预算超限停止，不偷用未来结果 |
| 同 Signal 两组合 | Engine 合同增量审后，最多 2 次 Runtime / Evaluation | 不启动额外 fit / predict / Feature / supplier；比较 Signal 原身份 |
| UI / HTML | owner Reader 一次；离线 render / 内容核对 | 不在开页或切页计算业务；Pages 由原流程交付 |

全教学一个重进程，拟 RSS 硬上限 6 GiB、外部 own-PID guard 5.5 GiB 提前停止；单阶段最多 120 s、总业务预算 8 分钟、新产物累计最多 1 GiB。所有数字是待父审的上限，不是实测保证；Engine 组合公共增量未完成前不运行该对照。执行报告逐阶段写 CALCULATED / LOADED_VERIFIED / CACHE_HIT / HISTORICAL_EVIDENCE / BLOCKED。

weekly 测量后，五年规划只写：唯一日期冷准备 + 各实际周成熟/矩阵/训练/推理/证明 I/O 累计 + 账户/展示另列；给范围、假设与未证明的历史 union / 覆盖成本，不直接乘单次 fit 毫秒。

字段规模由 registry 自动选择 OHLC/amount/factor（六个依赖字段）以及独立成员/日历证明，不增加临时特征。预计至多两次 weekly 业务训练/推理与八次公共 label-outcome 查询；没有供应商调用。若固定 Snapshot 覆盖、fit 成熟性、前 session anchor 或完整截面 parent 无法证明，记 BLOCKED，不以未来 outcome 或工作日补齐。

[完整候选方案与停止规则](../docs/design/05_axiom_research.md#ml-engineering-review-proposals)由父任务先审。当前公开 saved-Feature helper 不能接不同 config，weekly 样本是待审有界编排，不冒充正式 public rolling release。各计时重叠范围明确记录，五年规模不从单 fit 时间或旧 monthly 情景直接外推。
